In [15]:
import numpy as np
from collections import defaultdict

In [22]:
class Graph:
    
    def __init__(self):
        self.nodes = []
        self.adj = defaultdict(list)
        
        self.phi = {}
        self.psi = {}
    
    def add_node(self, node, unary):
        if node in self.nodes:
            raise ValueError(f'Node {node} already exists.')
        self.nodes.append(node)
        self.phi[node] = unary
    
    def add_edge(self, node1, node2, psi12):
        self.adj[node1].append(node2)
        self.adj[node2].append(node1)
        
        self.psi[(node1, node2)] = psi12
        self.psi[(node2, node1)] = psi12.T

In [ ]:
def tree_bp(tree, root):
    
    messages = {}
    
    def collect(node, parent):
        
        h = tree.phi[node]
        for neigh in tree.adj[node]:
            if neigh == parent:
                continue
            collect(neigh, node)
            h = h * messages[(neigh, node)]
        
        if parent is None:
            return
        msg = tree.psi[(node, parent)].T @ h
        msg /= msg.sum()
        messages[(node, parent)] = msg
    
    collect(root, None)
    
    marginals = {}
    
    def backward(node, parent):
        
        
        msgs = {}
        
        p = tree.phi[node]
        for neigh in tree.adj[node]:
            msg = messages[(neigh, node)]
            p = p * msg
            msgs[neigh] = msg
        
        p = p / p.sum()
        marginals[node] = p
        
        for neigh in tree.adj[node]:
            if neigh == parent:
                continue
            h = tree.phi[node]
            for name, msg in msgs.items():
                if name == neigh:
                    continue
                h = h * msg
        
            msg = tree.psi[(node, neigh)].T @ h
            msg /= msg.sum()
            messages[(node, neigh)] = msg
            
            backward(neigh, node)
    
    backward(root, None)
    
    return marginals

In [58]:
tree = Graph()
tree.add_node("A", [0.6, 0.4])
tree.add_node("B", [0.5, 0.5])
tree.add_node("C", [0.7, 0.3])

psi = np.array([[2,1],
                [1,2]])

tree.add_edge("A","B",psi)
tree.add_edge("B","C",psi)
tree.adj

defaultdict(list, {'A': ['B'], 'B': ['A', 'C'], 'C': ['B']})

In [59]:
tree_bp(tree, "A")

{'A': array([0.62114537, 0.37885463]), 'B': array([0.59911894, 0.40088106]), 'C': array([0.7092511, 0.2907489])}
